In [1]:
from langchain_community.document_loaders import TextLoader
# Document Loading:
loader_fruits = TextLoader("./notes/fruits_and_vegetables.txt")

documents_fruits = loader_fruits.load()

loader_diet = TextLoader("./notes/human_nutrition_and_dietary_requirements.txt")

documents_diet = loader_diet.load()

C:\Users\Yoga\AppData\Local\Temp\ipykernel_20260\817754794.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#  Text Splitting:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=1
)

chunks_fruits = text_splitter.split_documents(documents_fruits)
chunks_diet = text_splitter.split_documents(documents_diet)

In [3]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

#Embedding Generation, Vector Storage
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)
vectorstore_fruits = FAISS.from_documents(chunks_fruits, embedding_model)
vectorstore_fruits.save_local("./faiss_fruits")

vectorstore_diet = FAISS.from_documents(chunks_diet, embedding_model)
vectorstore_diet.save_local("./faiss_diet")

################################################################

# Retrieve both fruits and diet— load back and search
vectorstore_fruits = FAISS.load_local(
    "./faiss_fruits",
    embedding_model,
    allow_dangerous_deserialization=True
)
vectorstore_diet = FAISS.load_local(
    "./faiss_diet",
    embedding_model,
    allow_dangerous_deserialization=True
)


fruits_retriever = vectorstore_fruits.as_retriever(
    search_kwargs={"k": 2}
)
nutrition_retriever = vectorstore_diet.as_retriever(
    search_kwargs={"k": 2}
)

In [4]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="qwen2.5:1.5b", temperature=0.2)

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# Step 1: Define the Query Router prompt
router_prompt = ChatPromptTemplate.from_template("""
        You are a query router.

        Choose the most appropriate knowledge source for the user's question.

        Available sources:

        fruits:
        Information about fruits, vegetables, colours, seeds, plant parts,
        taste and cooking.

        nutrition:
        Information about vitamins, minerals, proteins, carbohydrates,
        fats, fibre, hydration and dietary requirements.

        Return ONLY one word: fruits or nutrition.

        User question: {question}
""")

# Step 2: Create the Query Router
query_router = router_prompt | llm | StrOutputParser()

In [6]:
# Step 2: Generate multiple alternative search queries
multi_query_prompt = ChatPromptTemplate.from_template("""
Generate 3 different search queries based on the user's question.
Each query should approach the same question from a different perspective.

Do not answer the question.
Return only the 3 queries, each on a separate line.

User question: {question}
""")

# Convert the prompt into an executable LCEL chain
multi_query_generator = multi_query_prompt | llm | StrOutputParser()

In [ ]:
# Step 3: Route the question and retrieve relevant documents
def route_and_retrieve(question):

    # Ask the LLM which knowledge source to use
    selected_route = query_router.invoke(
        {"question": question}
    ).strip().lower()

    print("Selected Route:", selected_route)

    # Select the appropriate retriever
    if selected_route == "fruits":
        selected_retriever = fruits_retriever

    elif selected_route == "nutrition":
        selected_retriever = nutrition_retriever

    else:
        raise ValueError("Invalid route selected: " + selected_route)

    # Generate multiple search queries from the original question
    generated_queries = multi_query_generator.invoke(
        {"question": question}
    )

    print("generated_queries are:", generated_queries)

     # Convert the generated queries into a Python list
    queries = [q.strip() for q in generated_queries.split("\n") if q.strip()]

    print("Generated Queries:", queries)

    # Retrieve documents for each generated query
    retrieved_docs = []

    for query in queries:
        retrieved_docs.extend(selected_retriever.invoke(query))

    # Remove duplicate documents based on their content
    unique_docs = []

    for doc in retrieved_docs:
        if doc.page_content not in [d.page_content for d in unique_docs]:
            unique_docs.append(doc)

     # Combine unique retrieved documents into context
    context = "\n\n".join(
        doc.page_content for doc in unique_docs
    )

    # Return both context and original question
    return {
        "context": context,
        "question": question
    }

In [10]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using the following context.

Context:
{context}

Question:
{question}
""")

# Step 4: Build the final RAG chain
chain = (
    RunnableLambda(route_and_retrieve)
    | prompt
    | llm
    | StrOutputParser()
)


# Step 5: Test
response = chain.invoke(
    "Difference between Fruits and diet"
)

print("Final response is : "+ response)

Selected Route: nutrition
Rewritten Query: 1. What are the nutritional differences between fruits and diets?
2. How do fruits compare to other food groups in terms of health benefits?
3. What are the health impacts of incorporating fruits into a balanced diet versus eating them as a standalone food?
Generated Queries: ['1. What are the nutritional differences between fruits and diets?', '2. How do fruits compare to other food groups in terms of health benefits?', '3. What are the health impacts of incorporating fruits into a balanced diet versus eating them as a standalone food?']
Final response is : Fruits and a balanced diet are closely related and complement each other in providing essential nutrients and maintaining overall health. Here are some key differences:

1. **Source of Nutrients**: Fruits are whole foods that are naturally rich in vitamins, minerals, fiber, and antioxidants. A balanced diet includes a variety of fruits, vegetables, whole grains, protein-rich foods, and hea